In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
from imblearn.over_sampling import SMOTE

df = pd.read_csv('high_risk_pregnancy_realistic_2000 (1).csv')

In [2]:
df.head()

,timestamp,patient_id,device_source,village,age_years,gravida_G,para_P,live_child_L,abortion_A,death_D,...,body_temperature_F,heart_rate_bpm,hemoglobin_g_dL,hba1c_percent,respiratory_rate_bpm,bmi,spo2_percent,edema_severity,symptoms_score_0_10,risk_class
0,2026-01-01 08:55:00,HRP202600001,ANM Tablet,Nandgaon,35,3,2,2,0,1,...,99.4,85.3,10.6,7.2,23.4,16.0,96.3,Mild,4.4,Risk
1,2026-01-01 09:00:00,HRP202600002,ASHA Mobile,Lakshmipur,33,5,0,0,0,0,...,97.9,94.6,9.1,5.4,19.5,26.2,99.5,NaN,2.3,No Risk
2,2026-01-01 09:17:00,HRP202600003,ASHA Mobile,Haripur,28,1,0,0,1,0,...,99.6,86.8,11.7,6.7,19.5,30.8,98.7,Severe,0.4,No Risk
3,2026-01-01 09:57:00,HRP202600004,ASHA Mobile,Shivpuri,25,2,0,0,0,0,...,97.5,101.0,NaN,6.0,16.9,22.3,97.2,Mild,2.0,No Risk
4,2026-01-01 10:06:00,HRP202600005,PHC Kiosk,Madanpur,27,2,0,0,0,0,...,98.6,102.8,NaN,5.3,25.1,21.6,95.9,NaN,3.7,No Risk


### Dropping cols

In [3]:
df=df.drop(columns=['timestamp','device_source']) #droping them bc smote only understand numeric data
df.head()

,patient_id,village,age_years,gravida_G,para_P,live_child_L,abortion_A,death_D,gestational_age_weeks,systolic_bp_mmHg,...,body_temperature_F,heart_rate_bpm,hemoglobin_g_dL,hba1c_percent,respiratory_rate_bpm,bmi,spo2_percent,edema_severity,symptoms_score_0_10,risk_class
0,HRP202600001,Nandgaon,35,3,2,2,0,1,28.8,112.7,...,99.4,85.3,10.6,7.2,23.4,16.0,96.3,Mild,4.4,Risk
1,HRP202600002,Lakshmipur,33,5,0,0,0,0,40.0,109.4,...,97.9,94.6,9.1,5.4,19.5,26.2,99.5,NaN,2.3,No Risk
2,HRP202600003,Haripur,28,1,0,0,1,0,29.7,132.6,...,99.6,86.8,11.7,6.7,19.5,30.8,98.7,Severe,0.4,No Risk
3,HRP202600004,Shivpuri,25,2,0,0,0,0,16.4,108.3,...,97.5,101.0,NaN,6.0,16.9,22.3,97.2,Mild,2.0,No Risk
4,HRP202600005,Madanpur,27,2,0,0,0,0,17.5,138.3,...,98.6,102.8,NaN,5.3,25.1,21.6,95.9,NaN,3.7,No Risk


### One hot Encoding on col village

In [4]:
df=pd.get_dummies(df,columns=['village'],drop_first=True)
# One-Hot Encoding of column village 
df.head()

,patient_id,age_years,gravida_G,para_P,live_child_L,abortion_A,death_D,gestational_age_weeks,systolic_bp_mmHg,diastolic_bp_mmHg,...,risk_class,village_Bhawanipur,village_Devnagar,village_Haripur,village_Khadar,village_Lakshmipur,village_Madanpur,village_Nandgaon,village_Rampur,village_Shivpuri
0,HRP202600001,35,3,2,2,0,1,28.8,112.7,83.0,...,Risk,False,False,False,False,False,False,True,False,False
1,HRP202600002,33,5,0,0,0,0,40.0,109.4,60.8,...,No Risk,False,False,False,False,True,False,False,False,False
2,HRP202600003,28,1,0,0,1,0,29.7,132.6,79.2,...,No Risk,False,False,True,False,False,False,False,False,False
3,HRP202600004,25,2,0,0,0,0,16.4,108.3,64.6,...,No Risk,False,False,False,False,False,False,False,False,True
4,HRP202600005,27,2,0,0,0,0,17.5,138.3,64.8,...,No Risk,False,False,False,False,False,True,False,False,False


In [5]:
df['edema_severity'].unique()

array(['Mild', nan, 'Severe', 'Moderate'], dtype=object)

In [6]:
df['edema_severity']=df['edema_severity'].fillna('None') # filling nan value as none
df['edema_severity']=df['edema_severity'].map({'None':0,'Mild':1,'Moderate':2,'Severe':3})
df['edema_severity']

0       1
1       0
2       3
3       1
4       0
       ..
1995    0
1996    0
1997    0
1998    2
1999    1
Name: edema_severity, Length: 2000, dtype: int64

In [7]:
df.isnull().sum()

patient_id                    0
age_years                     0
gravida_G                     0
para_P                        0
live_child_L                  0
abortion_A                    0
death_D                       0
gestational_age_weeks        41
systolic_bp_mmHg              0
diastolic_bp_mmHg             0
random_blood_sugar_mg_dL    122
body_temperature_F           58
heart_rate_bpm                0
hemoglobin_g_dL             101
hba1c_percent               161
respiratory_rate_bpm          0
bmi                          60
spo2_percent                 79
edema_severity                0
symptoms_score_0_10          63
risk_class                    0
village_Bhawanipur            0
village_Devnagar              0
village_Haripur               0
village_Khadar                0
village_Lakshmipur            0
village_Madanpur              0
village_Nandgaon              0
village_Rampur                0
village_Shivpuri              0
dtype: int64

In [8]:
df['risk_class']=df['risk_class'].map({'No Risk':0,'Risk':1})
print(df['risk_class'].value_counts())

risk_class
0    1696
1     304
Name: count, dtype: int64


### Train And Test split


In [9]:
x=df.drop(columns='risk_class') # x-> features
y=df['risk_class']  # y->target

### Train/Test Split (Group Aware)

In [10]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=20)

train_idx, test_idx = next(gss.split(x, y, groups=df["patient_id"]))

x_train, x_test = x.iloc[train_idx], x.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

# Confirm no patient leaked across both sets
train_patients = set(df["patient_id"].iloc[train_idx])
test_patients = set(df["patient_id"].iloc[test_idx])

print("Patient overlap between train & test:", train_patients.intersection(test_patients))

print("\nTrain class distribution:")
print(y_train.value_counts(normalize=True) * 100)

print("\nTest class distribution:")
print(y_test.value_counts(normalize=True) * 100)

Patient overlap between train & test: set()

Train class distribution:
risk_class
0    85.321674
1    14.678326
Name: proportion, dtype: float64

Test class distribution:
risk_class
0    82.706767
1    17.293233
Name: proportion, dtype: float64


### Dropping col patient_id from both test and train

In [11]:
x_train = x_train.drop(columns=["patient_id"])
x_test = x_test.drop(columns=["patient_id"])

In [12]:
num_cols = ['gestational_age_weeks', 'random_blood_sugar_mg_dL',
            'body_temperature_F', 'hemoglobin_g_dL', 'hba1c_percent',
            'bmi', 'spo2_percent', 'symptoms_score_0_10']

train_medians = {}  # save so you can reuse on future/new data
for col in num_cols:
    median_val = x_train[col].median()
    train_medians[col] = median_val
    x_train[col] = x_train[col].fillna(median_val)
    x_test[col]  = x_test[col].fillna(median_val)

print("Train medians used:", train_medians)
print("\nRemaining NaNs in train:", x_train.isnull().sum().sum())
print("Remaining NaNs in test: ", x_test.isnull().sum().sum())

Train medians used: {'gestational_age_weeks': 24.2, 'random_blood_sugar_mg_dL': 114.0, 'body_temperature_F': 98.4, 'hemoglobin_g_dL': 10.8, 'hba1c_percent': 5.4, 'bmi': 24.9, 'spo2_percent': 97.8, 'symptoms_score_0_10': 2.8}

Remaining NaNs in train: 0
Remaining NaNs in test:  0


### Smote implemenation

In [13]:
smote=SMOTE(random_state=20)
x_train_smote,y_train_smote=smote.fit_resample(x_train,y_train)
print("Before smote : ")
print(y_train.value_counts())
print("After smote : ")
print(y_train_smote.value_counts())

Before smote : 
risk_class
0    1366
1     235
Name: count, dtype: int64
After smote : 
risk_class
1    1366
0    1366
Name: count, dtype: int64


In [14]:
print('Train shape : ',x_train_smote.shape)
print('Train shape : ',x_test.shape)

Train shape :  (2732, 28)
Train shape :  (399, 28)


In [15]:
from sklearn.preprocessing import RobustScaler

scaler = RobustScaler()
x_train_smote = scaler.fit_transform(x_train_smote)  # fit on smote data
x_test_scaled = scaler.transform(x_test)              # transform test

print('Scaling done!')

Scaling done!


In [22]:
import joblib

joblib.dump(x_train_smote, 'x_train_smote.pkl')
joblib.dump(y_train_smote, 'y_train_smote.pkl')
joblib.dump(x_test_scaled, 'x_test_scaled.pkl')  
joblib.dump(y_test,        'y_test.pkl')

['y_test.pkl']

In [17]:
print(y_test.value_counts())

risk_class
0    330
1     69
Name: count, dtype: int64


In [18]:
print(y_train_smote.value_counts())

risk_class
1    1366
0    1366
Name: count, dtype: int64


In [20]:
smote_test = SMOTE(random_state=42)
x_test_smote, y_test_smote = smote_test.fit_resample(x_test_scaled, y_test)

print(y_test_smote.value_counts())


risk_class
0    330
1    330
Name: count, dtype: int64


In [23]:
joblib.dump(x_test_smote, 'x_test_smote.pkl')
joblib.dump(y_test_smote, 'y_test_smote.pkl')
print("saved")

saved
